# Topic 3 — Quantum Circuits, State-Vector Simulation, AI-assited coding, and the Trotter Algorithm

As we saw in the previous modules, the difficulty of classically simulating quantum mechanics with many degrees of freedom comes from its exponential $\sim 2^n$ resource requirements. Paraphrasing Richard Feynman's remark, "The best way to simulate a quantum mechanical system is to use another quantum mechanical system" — this is the founding idea of quantum computation. After several decades of effort, we still do not have a fault-tolerant quantum computer that can operate to the error tolerance required for practical advantage. Nevertheless, significant progress has been made: we now have machines that can carry out quantum circuits on the order of 100 qubits or more, though with considerable noise and error. These devices are dubbed **NISQ** (Noisy Intermediate-Scale Quantum) machines, a term coined by John Preskill in his 2018 paper ["Quantum Computing in the NISQ Era and Beyond"](https://quantum-journal.org/papers/q-2018-08-06-79/).

The accessibility of these machines is still limited, and it is therefore important to be able to *simulate* quantum circuits classically, ensuring the correctness of a quantum algorithm before running it on real hardware.



---

## 1. Quantum circuits

A **quantum circuit** is a model of quantum computation in which a sequence of quantum gates is applied to an $n$-qubit register, transforming an initial state into a final state that can then be measured.

#### 1.1 Qubits and state space

A single qubit lives in $\mathbb{C}^2$ and is written
$$
|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad |\alpha|^2 + |\beta|^2 = 1.
$$
An $n$-qubit register lives in $(\mathbb{C}^2)^{\otimes n} \cong \mathbb{C}^{2^n}$, spanned by the computational basis states $|b_{n-1}\cdots b_1 b_0\rangle$ for $b_j \in \{0,1\}$.

#### 1.2 Quantum gates

A quantum gate is a **unitary** operation $U$ (so that $UU^\dagger = I$). Gates are the quantum analog of logic gates; the key difference is that they are reversible and can act on superpositions.

**Common single-qubit gates:**

| Gate | Matrix | Action |
|------|--------|--------|
| $X$ (NOT) | $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ | Bit flip: $|0\rangle\leftrightarrow|1\rangle$ |
| $Z$ | $\begin{pmatrix}1&0\\0&-1\end{pmatrix}$ | Phase flip: $|1\rangle\mapsto-|1\rangle$ |
| $H$ (Hadamard) | $\frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ | $|0\rangle\mapsto|{+}\rangle,\ |1\rangle\mapsto|{-}\rangle$ |
| $R_x(\theta)$ | $\begin{pmatrix}\cos\frac{\theta}{2}&-i\sin\frac{\theta}{2}\\-i\sin\frac{\theta}{2}&\cos\frac{\theta}{2}\end{pmatrix}$ | Rotation about $x$-axis by $\theta$ |
| $R_z(\theta)$ | $\begin{pmatrix}e^{-i\theta/2}&0\\0&e^{i\theta/2}\end{pmatrix}$ | Rotation about $z$-axis by $\theta$ |

**Common two-qubit gates:**

| Gate | Action |
|------|--------|
| CNOT | Flips the target qubit if the control qubit is $|1\rangle$ |
| $CZ$ | Applies a $Z$ phase to the target if the control is $|1\rangle$ |
| $R_{zz}(\theta) = e^{-i\frac{\theta}{2}Z\otimes Z}$ | Two-qubit rotation, building block of Trotter circuits |

A gate $U$ acting on qubit $j$ of an $n$-qubit register corresponds to the $2^n\times 2^n$ unitary
$$
I^{\otimes(n-1-j)} \otimes U \otimes I^{\otimes j}.
$$

#### 1.3 Circuit diagrams

We often use the circuit diagram to represent the *quantum circuit*, which is an instruction of the sequence of quantum gates for the machine to apply.

![caption](./quantumcircuit.svg)

Circuits are read left to right; each horizontal wire is a qubit, and each box or symbol is a gate:

A circuit of $L$ gates maps the initial state as
$$
|\psi_{\rm out}\rangle = U_L \cdots U_2\, U_1\,|00\ldots0\rangle~.
$$
Note we always assume the initial state of the circuit to be $|00\ldots0\rangle$.

We will first build a "quick-and-dirty" state-vector simulator ourselves, serving as the ground truth. We will then use AI to help us build a more efficient simulator, testing it against our quick-and-dirty simulator for correctness.

The `QuantumCircuit` class in the `qiskit` package has become a fairly standard way of constructing quantum circuits.

## Exercise 1: State-vector simulator

### Convention

We use the same **big-endian** convention as in the previous notebooks:
basis state $|b_0 b_1 \cdots b_{n-1}\rangle$ is stored at index
$$
a = \sum_{j=0}^{n-1} b_j\, 2^j, \qquad b_j \in \{0,1\}.
$$
A two-qubit gate $U$ acting on qubits $j < k$ therefore has matrix elements in the subspace ordered as $|00\rangle, |10\rangle, |01\rangle, |11\rangle$ (integer order with qubit $j$ as the least significant bit of the subspace index):
$$
U = \begin{pmatrix}
u_{00,00} & u_{00,10} & u_{00,01} & u_{00,11} \\
u_{10,00} & u_{10,10} & u_{10,01} & u_{10,11} \\
u_{01,00} & u_{01,10} & u_{01,01} & u_{01,11} \\
u_{11,00} & u_{11,10} & u_{11,01} & u_{11,11}
\end{pmatrix}
$$
where the subscript pair $ab$ denotes $b_j = a$, $b_k = b$.

Note on `np.kron`: `np.kron(A, B)` computes $A \otimes B$, where $B$ acts on the **lower**-index qubits (smaller $j$, less significant bits) and $A$ acts on the **higher**-index qubits. Accordingly, a gate $U$ on qubit $j$ of an $n$-qubit register is embedded as
$$
\underbrace{I^{\otimes(n-1-j)}}_{\text{qubits } n{-}1,\ldots,j{+}1} \otimes\; U \;\otimes\; \underbrace{I^{\otimes j}}_{\text{qubits } j{-}1,\ldots,0}.
$$

---

### Tasks

**1.** Implement `apply_gate_1q(state, U, j)` that applies a $2\times 2$ unitary $U$ to qubit $j$ of the state vector `state` (length $2^n$). Use `np.kron` and `np.eye(2**m)` for $I^{\otimes m}$.

**2.** Implement `apply_gate_2q(state, U, j, k)` that applies a $4\times 4$ unitary $U$ to adjacent qubits $j < k = j+1$. The matrix $U$ is in the convention above.
(How would you handle non-adjacent $k \neq j+1$?)

**3.** Write `simulate(gates, n)` that starts from $|00\cdots0\rangle$ and applies a list of gates in sequence, where each gate is a tuple `(U, [qubit_labels])`.

**4.** Verify your simulator by preparing the two-qubit state
$$
|\psi\rangle = \tfrac{1}{\sqrt{2}}\bigl(|00\rangle + i\,|11\rangle\bigr).
$$
The circuit is: apply $H$ to qubit 0, then $S$ to qubit 0, then CNOT with control = qubit 0 and target = qubit 1. Check that the amplitude vector equals $[\tfrac{1}{\sqrt{2}},\, 0,\, 0,\, \tfrac{i}{\sqrt{2}}]$.

## Exercise 2: Build a More Efficient State-Vector Simulator Using AI

Use an AI coding assistant to build a more efficient state-vector simulator.

Document the following:

1. Prompting strategies. Describe any useful practices, techniques, or prompt patterns that helped you obtain better results from the AI assistant.

2. Instructions and supporting files. Include any Markdown files or other documents that you used to guide the AI agent. These instructions should clearly specify the desired functionality of the simulator. For example, describe:

- the functions or classes you want the AI to implement;

- the expected inputs and outputs;

- coding, naming, or formatting conventions;

- performance or memory requirements;

- any restrictions on external libraries.

3. Verification and debugging. Describe the methods you used to verify the correctness of the AI-generated code. Discuss any useful practices for identifying errors, testing individual components, and checking the final results. 

4. Document any **failure case**: an example of AI-generated code that initially looked acceptable but was incorrect, inefficient, or poorly designed, together with an explanation of how you detected and corrected it.

### Reference solution notes

**Why the naive simulator is slow.**
`apply_gate_1q` builds a $2^n \times 2^n$ matrix via `np.kron` and multiplies it with the state: $O(4^n)$ work per gate. For $n = 20$ that is already $\sim 10^{12}$ floating-point operations.

**The efficient idea.**
A gate on qubit $j$ only couples pairs of amplitudes that differ in bit $j$. If we reshape the flat state vector into an $n$-dimensional tensor `psi` of shape `(2,2,...,2)`, with qubit $j$ mapped to tensor axis $n{-}1{-}j$ (C-order), then applying $U$ is a single `tensordot` contraction along that axis — $O(2^n)$ work, no large matrix constructed.

A two-qubit gate on adjacent $(j, k{=}j{+}1)$ mixes groups of four amplitudes. We gather all $2^{n-2}$ such groups with vectorized bit-manipulation and apply $U$ as a batched $4 \times 4$ matrix–vector product — again $O(2^n)$.

---

**Example spec given to AI (prompting tip: be explicit about the convention):**

```
# Spec: efficient state-vector simulator

Convention (must match Exercise 1):
- Basis state |b_0 b_1 ... b_{n-1}> stored at integer index a = sum_j b_j * 2^j.
- Qubit j = bit j of a (LSB = qubit 0).
- 2-qubit gate U (4x4): rows/cols ordered by subspace index b_j + 2*b_k
  i.e., |00>=0, |10>=1, |01>=2, |11>=3.

Functions to implement:
  apply_gate_1q_fast(state, U, j)      # 2x2 gate, no np.kron allowed
  apply_gate_2q_fast(state, U, j, k)   # 4x4 gate, adjacent qubits only (k = j+1)
  simulate_fast(gates, n)              # same interface as simulate() in Exercise 1

Correctness requirement:
  np.allclose(simulate_fast(circuit, n), simulate(circuit, n)) == True
  for any valid circuit.
```

**Verification trick:** generate a random circuit, run both simulators, and assert `np.allclose`. Then use `%timeit` to see the speedup.

## 3. Trotter algorithm

Now suppose you have a quantum computer that can carry out quantum circuits composed of one- and two-qubit gates. How would you calculate
$$
|\psi(t)\rangle = e^{-iHt}|0\cdots 0\rangle?
$$
Seth Lloyd proposed a remarkably simple quantum simulation algorithm based on the **Trotter--Suzuki decomposition** [S. Lloyd, *Universal Quantum Simulators*, Science **273**, 1073 (1996)](https://www.science.org/doi/10.1126/science.273.5278.1073)---a technique that physicists have long used in numerical methods such as quantum Monte Carlo. The basic idea is to decompose the time evolution generated by a complicated Hamiltonian into a sequence of short-time evolutions generated by simpler terms.

As we will see in this section, and later in the accompanying notebook, the same idea can also be used as a classical numerical method for calculating quantum dynamics.

### 3.1 Trotter decomposition

Suppose the Hamiltonian can be written as a sum of simpler terms,
$$
H = \sum_{\alpha=1}^{M} H_\alpha,
$$
where each $H_\alpha$ acts only on a small number of qubits. For example, for a local spin Hamiltonian, each $H_\alpha$ may be a one- or two-qubit operator.

If all the terms commute,
$$
[H_\alpha,H_\beta]=0,
$$
then the time-evolution operator factorizes exactly:
$$
e^{-iHt}=\prod_{\alpha=1}^{M}e^{-iH_\alpha t}.
$$

In general, however, the terms in the Hamiltonian do not commute, so this factorization is not exact. The Trotter formula tells us that it becomes approximately correct over a sufficiently short time interval $\Delta t$:
$$
e^{-iH\Delta t}
=
e^{-iH_1\Delta t}
e^{-iH_2\Delta t}
\cdots
e^{-iH_M\Delta t}
+
O(\Delta t^2).
$$

We can therefore divide the total evolution time $t$ into $N$ small steps,
$$
\Delta t = \frac{t}{N},
$$
and approximate
$$
e^{-iHt}
=
\left(e^{-iH\Delta t}\right)^N
\approx
\left(
e^{-iH_1\Delta t}
e^{-iH_2\Delta t}
\cdots
e^{-iH_M\Delta t}
\right)^N.
$$

This is known as the **first-order Trotter formula**.

The approximation becomes exact in the limit
$$
N\rightarrow\infty,
\qquad
\Delta t\rightarrow 0.
$$

The origin of the error can already be seen for a Hamiltonian with two terms,
$$
H=A+B.
$$
Using the Baker--Campbell--Hausdorff formula,
$$
e^{-iA\Delta t}e^{-iB\Delta t}
=
e^{-i(A+B)\Delta t
-\frac{1}{2}[A,B]\Delta t^2+\cdots}.
$$
Thus, the error is controlled by the noncommutativity of the Hamiltonian terms. If $[A,B]=0$, the Trotter decomposition is exact even for finite $\Delta t$.

For a fixed total evolution time $t$, the accumulated error of the first-order Trotter formula scales roughly as
$$
O\left(\frac{t^2}{N}\right),
$$
with coefficients determined by commutators such as $[H_\alpha,H_\beta]$.

### 3.2 Second-order Trotter decomposition

A more accurate approximation is obtained by symmetrizing the decomposition. For
$$
H=A+B,
$$
the second-order Trotter--Suzuki formula is
$$
e^{-i(A+B)\Delta t}
\approx
e^{-iA\Delta t/2}
e^{-iB\Delta t}
e^{-iA\Delta t/2}.
$$

The error per time step is now $O(\Delta t^3)$ rather than $O(\Delta t^2)$. Repeating the step $N$ times therefore gives
$$
e^{-iHt}
\approx
\left[
e^{-iA\Delta t/2}
e^{-iB\Delta t}
e^{-iA\Delta t/2}
\right]^N,
$$
with a total error scaling roughly as
$$
O\left(\frac{t^3}{N^2}\right).
$$

For a Hamiltonian containing many terms, the same idea can be generalized to
$$
e^{-iH\Delta t}
\approx
e^{-iH_1\Delta t/2}
e^{-iH_2\Delta t/2}
\cdots
e^{-iH_M\Delta t}
\cdots
e^{-iH_2\Delta t/2}
e^{-iH_1\Delta t/2}.
$$

Higher-order Suzuki formulas can reduce the Trotter error even further, although they require more exponentials, and therefore deeper quantum circuits.

### 3.3 From Hamiltonian terms to quantum gates

Why is this useful on a quantum computer?

Suppose, for example, that the Hamiltonian contains a term
$$
H_\alpha = J Z_i Z_j.
$$
Its short-time evolution is
$$
e^{-iJ\Delta t Z_iZ_j}.
$$

Although $e^{-iHt}$ for the full many-body Hamiltonian may be extremely complicated, the operator
$$
e^{-iJ\Delta t Z_iZ_j}
$$
acts on only two qubits and can therefore be implemented using a short sequence of elementary quantum gates.

Similarly, a transverse-field term
$$
H_\alpha = hX_i
$$
gives
$$
e^{-ih\Delta t X_i},
$$
which is simply a single-qubit rotation.

As a concrete example, consider the transverse-field Ising model
$$
H
=
-J\sum_i Z_iZ_{i+1}
-h\sum_i X_i.
$$

We can separate it into
$$
H_Z=-J\sum_i Z_iZ_{i+1},
\qquad
H_X=-h\sum_i X_i.
$$

A first-order Trotter step is then
$$
e^{-iH\Delta t}
\approx
e^{-iH_Z\Delta t}
e^{-iH_X\Delta t}.
$$

Since all the $Z_iZ_{i+1}$ terms commute with one another,
$$
e^{-iH_Z\Delta t}
=
\prod_i e^{iJ\Delta t Z_iZ_{i+1}},
$$
and similarly,
$$
e^{-iH_X\Delta t}
=
\prod_i e^{ih\Delta t X_i}.
$$

Thus, one Trotter step can be implemented entirely using one- and two-qubit gates. Repeating this circuit $N$ times approximates the many-body evolution up to time
$$
t=N\Delta t.
$$

Schematically,
$$
|\psi(t)\rangle
=
e^{-iHt}|\psi(0)\rangle
\approx
\left[
\prod_{\alpha}
e^{-iH_\alpha\Delta t}
\right]^N
|\psi(0)\rangle.
$$

This provides a direct translation from a local Hamiltonian into a quantum circuit.

### 3.4 Trotterization as a classical algorithm

It is worth emphasizing that Trotter decomposition itself is not inherently a quantum-computing algorithm. It is a decomposition of the time-evolution operator.

On a quantum computer, we implement each factor $e^{-iH_\alpha\Delta t}$ as a quantum gate.

On a classical computer, however, we can represent the wavefunction explicitly and apply exactly the same sequence of local operators to it. For a system of $n$ qubits, the state vector contains
$2^n$ complex amplitudes, so this approach still requires exponentially large memory. Nevertheless, applying local gates directly to a state vector can often be much more efficient than explicitly constructing the full $2^n\times 2^n$ Hamiltonian matrix and computing $e^{-iHt}$.

In this sense, a Trotterized quantum circuit is also a natural classical algorithm for simulating quantum dynamics.

This observation will be useful later: instead of thinking of quantum time evolution only as a matrix exponential, we can think of it as a sequence of **local updates** acting on the many-body wavefunction.



## Exercise 3: Trotter algorithm for the mixed-field Ising model

**1.** Write a function to generate the second-order Trotter circuit for the mixed-field Ising model with periodic boundary conditions (PBC):
$$
H
=
-J\sum_{j=0}^{n-1}X_jX_{j+1}
-h_z\sum_{j=0}^{n-1}Z_j
-h_x\sum_{j=0}^{n-1}X_j~,
$$
where $j+n \equiv j$. The function should take the Hamiltonian parameters $(J,\, h_z,\, h_x)$, number of qubits $n$, Trotter step size $\Delta t$, and number of Trotter steps $N_T$ as input, and return the full gate sequence in the format accepted by your state-vector simulator. The total simulated time is $T = N_T \cdot \Delta t$.

**2.** Choose a set of parameters $(J,\, h_z,\, h_x)$, fix $n = 10$, and compute $\langle Z_0 \rangle(t)$ for several values of total time $T$ using the second-order Trotter decomposition with several different step sizes $\Delta t$. Compare with the exact result via matrix exponentiation you developed previously on a plot.